# XOR Expressive Power Practice

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umyunsang/edu/blob/main/ComputerScience/03_ai-ml-data/quantum-ml/1.quantum-ml-overview/expressive-power-limit/notebooks/practice/2_xor_expressive_power_practice.ipynb)

이 노트북은 `2. 표현력의 한계.pdf`의 XOR 문제 실습을 Colab 실행용으로 정리한 버전입니다.

**대상**
- 선형 모델이 표현하지 못하는 패턴을 직접 확인하려는 학습자
- QML 모델을 다루기 전에 classical baseline의 표현력 한계를 이해하려는 학습자

**학습 목표**
- XOR 데이터가 왜 직선 하나로 분리되지 않는지 시각화한다.
- Logistic Regression이 XOR 패턴을 완벽하게 학습하지 못하는 것을 확인한다.
- interaction feature를 추가하면 선형 모델의 표현력이 어떻게 달라지는지 비교한다.
- 비선형 모델이 같은 문제를 어떻게 해결하는지 확인한다.


## Outline

1. Import libraries
2. Create XOR data
3. Visualize the XOR pattern
4. Train Logistic Regression on raw features
5. Visualize the linear decision boundary
6. Add an interaction feature
7. Compare with a non-linear SVM
8. Exercises


## 1. Import libraries

Colab 기본 런타임에 포함된 패키지만 사용합니다. 별도 설치 셀은 두지 않습니다.


In [ ]:
from __future__ import annotations

import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC


## 2. Create XOR data

XOR은 두 입력이 서로 다를 때 `1`, 같을 때 `0`이 되는 데이터입니다. 같은 클래스가 대각선 방향에 놓이기 때문에 직선 하나로 두 클래스를 완벽히 나누기 어렵습니다.


In [ ]:
X = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1],
])
y = np.array([0, 1, 1, 0])

print('XOR table')
print('X1 X2 | Y')
for features, label in zip(X, y):
    print(f'{features[0]}  {features[1]}  | {label}')


## 3. Visualize the XOR pattern

`Y=1`과 `Y=0`이 각각 대각선에 놓이는지 확인합니다. 이 배치가 선형 모델의 표현력 한계를 만드는 핵심입니다.


In [ ]:
plt.figure(figsize=(5, 5))

for index in range(len(X)):
    if y[index] == 0:
        plt.scatter(
            X[index][0],
            X[index][1],
            marker='x',
            s=150,
            linewidths=3,
            label='Y=0' if index == 0 else '',
        )
    else:
        plt.scatter(
            X[index][0],
            X[index][1],
            marker='o',
            s=150,
            label='Y=1' if index == 1 else '',
        )

plt.xlabel('X1')
plt.ylabel('X2')
plt.title('XOR Data')
plt.xticks([0, 1])
plt.yticks([0, 1])
plt.legend()
plt.grid(True)
plt.show()


## 4. Train Logistic Regression on raw features

Logistic Regression은 선형 결정 경계를 학습합니다. XOR 데이터는 선형 분리 불가능하므로 예측이 완벽하지 않습니다.


In [ ]:
log_model = LogisticRegression()
log_model.fit(X, y)

y_pred = log_model.predict(X)
raw_accuracy = accuracy_score(y, y_pred)

print('예측 결과:', y_pred)
print('정답:', y)
print('정확도:', raw_accuracy)


## 5. Visualize the linear decision boundary

배경색은 Logistic Regression의 예측 영역입니다. 직선 하나로 나누기 때문에 일부 점을 잘못 분류할 수밖에 없습니다.


In [ ]:
x1_min, x1_max = -0.25, 1.25
x2_min, x2_max = -0.25, 1.25
xx, yy = np.meshgrid(
    np.linspace(x1_min, x1_max, 200),
    np.linspace(x2_min, x2_max, 200),
)
grid = np.c_[xx.ravel(), yy.ravel()]
grid_pred = log_model.predict(grid).reshape(xx.shape)

plt.figure(figsize=(5, 5))
plt.contourf(xx, yy, grid_pred, alpha=0.25, levels=[-0.5, 0.5, 1.5])

for label, marker in [(0, 'x'), (1, 'o')]:
    points = X[y == label]
    plt.scatter(
        points[:, 0],
        points[:, 1],
        marker=marker,
        s=150,
        linewidths=3,
        label=f'Y={label}',
    )

plt.xlabel('X1')
plt.ylabel('X2')
plt.title('Logistic Regression on XOR')
plt.xticks([0, 1])
plt.yticks([0, 1])
plt.legend()
plt.grid(True)
plt.show()


## 6. Add an interaction feature

PDF의 결론처럼 XOR을 해결하려면 비선형 모델 또는 새로운 feature가 필요합니다. 여기서는 `X1 * X2` interaction feature를 추가해 선형 모델이 더 풍부한 표현을 갖도록 만듭니다.


In [ ]:
X_with_interaction = np.column_stack([
    X,
    X[:, 0] * X[:, 1],
])

feature_model = LogisticRegression()
feature_model.fit(X_with_interaction, y)

feature_pred = feature_model.predict(X_with_interaction)
feature_accuracy = accuracy_score(y, feature_pred)

print('새 feature: X1 * X2')
print('확장된 입력')
print(X_with_interaction)
print('예측 결과:', feature_pred)
print('정답:', y)
print('정확도:', feature_accuracy)


## 7. Compare with a non-linear SVM

같은 XOR 데이터에 비선형 RBF kernel을 쓰는 SVM을 적용해 봅니다. 입력 feature를 직접 늘리지 않아도 모델 자체가 비선형 결정 경계를 만들 수 있습니다.


In [ ]:
nonlinear_model = SVC(kernel='rbf', gamma='scale')
nonlinear_model.fit(X, y)

nonlinear_pred = nonlinear_model.predict(X)
nonlinear_accuracy = accuracy_score(y, nonlinear_pred)

print('RBF SVM 예측 결과:', nonlinear_pred)
print('정답:', y)
print('정확도:', nonlinear_accuracy)


## 8. Summary

- XOR 데이터는 직선 하나로 완벽히 분리할 수 없습니다.
- Logistic Regression 같은 선형 모델은 raw XOR feature만으로는 표현력이 부족합니다.
- interaction feature를 추가하거나 비선형 모델을 사용하면 XOR 패턴을 더 잘 표현할 수 있습니다.
- 이후 QML 실습에서도 모델의 성능을 보기 전에 데이터 표현과 feature map의 역할을 먼저 확인해야 합니다.


## Exercises

1. `X1 * X2` 대신 `X1 + X2` feature만 추가하면 정확도가 어떻게 달라지는지 확인하세요.
2. `SVC(kernel='linear')`와 `SVC(kernel='rbf')`의 결과를 비교하세요.
3. XOR 입력을 `[-1, 1]` 스케일로 바꾼 뒤 Logistic Regression 결과가 달라지는지 확인하세요.
